In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "MCQ_D.xpt"
csv_file = DATA_DIR / "MCQ_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched source copy for final fidelity checking
source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 MCQ_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "MCQ010",
    "MCQ025",
    "MCQ035",
    "MCQ040",
    "MCQ050",
    "MCQ053",
    "MCQ080",
    "MCQ092",
    "MCD093",
    "MCQ140",
    "MCQ149",
    "MCQ150G",
    "MCQ150Q",
    "MCQ160A",
    "MCQ160B",
    "MCQ160C",
    "MCQ160D",
    "MCQ160E",
    "MCQ160F",
    "MCQ160G",
    "MCQ160K",
    "MCQ160L",
    "MCQ160M",
    "MCQ170K",
    "MCQ170L",
    "MCQ170M",
    "MCQ180A",
    "MCQ180B",
    "MCQ180C",
    "MCQ180D",
    "MCQ180E",
    "MCQ180F",
    "MCQ180G",
    "MCQ180K",
    "MCQ180L",
    "MCQ180M",
    "MCQ190",
    "MCQ220",
    "MCQ230A",
    "MCQ230B",
    "MCQ230C",
    "MCQ230D",
    "MCQ240A",
    "MCQ240B",
    "MCQ240C",
    "MCQ240D",
    "MCQ240E",
    "MCQ240F",
    "MCQ240G",
    "MCQ240H",
    "MCQ240I",
    "MCQ240J",
    "MCQ240K",
    "MCQ240L",
    "MCQ240M",
    "MCQ240N",
    "MCQ240O",
    "MCQ240P",
    "MCQ240Q",
    "MCQ240R",
    "MCQ240S",
    "MCQ240T",
    "MCQ240U",
    "MCQ240V",
    "MCQ240W",
    "MCQ240X",
    "MCQ240Y",
    "MCQ240Z",
    "MCQ240AA",
    "MCQ240BB",
    "MCQ240CC",
    "MCQ240DD",
    "MCQ240DK",
    "MCQ245A",
    "MCQ245B",
    "MCQ265",
    "MCQ268A",
    "MCQ268B",
    "MCQ268C",
    "MCQ268D",
    "MCQ300A",
    "MCQ300B",
    "MCQ300C",
    "MCQ310",
    "MCQ320",
    "MCD330",
    "MCQ340",
    "MCQ350"
]


if len(df) != 9822:
    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 89:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "MCQ_D columns/order differ.\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: MCQ_D structure = "
    "9,822 rows x 89 columns."
)


# ============================================================
# 4. KNOWN SAS/XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{int(true_zero_counts.sum()):,}"
)

print(
    "Tiny artifact occurrences:",
    f"{int(tiny_counts.sum()):,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 5. EXPECTED VALIDATED TINY COUNTS
# ============================================================

EXPECTED_TINY_COUNTS = {

    "MCQ150Q": 1028,

    "MCQ180B": 1,

    "MCQ180L": 1,

    "MCQ245B": 2061,

    "MCQ340": 48
}


EXPECTED_TINY_TOTAL = 3139


# ============================================================
# 6. VALIDATE TINY ARTIFACT LOCATIONS
# ============================================================

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)


expected_tiny_columns = set(
    EXPECTED_TINY_COUNTS.keys()
)


unexpected_columns = (
    actual_tiny_columns
    -
    expected_tiny_columns
)


if unexpected_columns:

    raise ValueError(
        "STOP: Tiny values found in unexpected "
        "MCQ_D variables:\n"
        + str(sorted(unexpected_columns))
    )


missing_expected_columns = (
    expected_tiny_columns
    -
    actual_tiny_columns
)


if missing_expected_columns:

    raise ValueError(
        "STOP: Expected tiny-value variables missing:\n"
        + str(sorted(missing_expected_columns))
    )


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        tiny_counts[col]
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


if int(
    tiny_counts.sum()
) != EXPECTED_TINY_TOTAL:

    raise ValueError(
        "Unexpected total tiny-value count."
    )


print(
    "\nPASS: Exactly 3,139 validated "
    "tiny artifacts found."
)


# ============================================================
# 7. RESTORE VALIDATED ZEROS
# ============================================================

print("\n--- ZERO RESTORATION ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    count = int(
        mask.sum()
    )


    df.loc[
        mask,
        col
    ] = 0.0


    print(
        f"{col:<10}: "
        f"{count:,} tiny -> 0"
    )


# ============================================================
# 8. VERIFY RESTORED ZERO COUNTS
# ============================================================

print("\n--- RESTORED ZERO COUNTS ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col:<10}: "
        f"{zero_count:,}"
    )


    if zero_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} zeros, "
            f"found {zero_count:,}."
        )


print(
    "PASS: All 3,139 validated zeros restored."
)


# ============================================================
# 9. VERIFY NO TINY ARTIFACT REMAINS
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nRemaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny-value artifact remains."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 10. CONFIRM NO GENUINE DECIMAL VARIABLES
# ============================================================

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_failures = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    if count > 0:

        print(
            f"{col:<10}: "
            f"fractional values = {count:,}"
        )


        fractional_failures.append(
            col
        )


if fractional_failures:

    raise ValueError(
        "STOP: Genuine decimal values found in:\n"
        + str(fractional_failures)
    )


print(
    "PASS: MCQ_D contains no genuine decimal variables."
)


# ============================================================
# 11. SEQN VALIDATION
# ============================================================

print("\n--- SEQN CHECK ---")


seqn_values = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn_values,
        np.round(seqn_values),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


seqn_missing = int(
    df["SEQN"]
    .isna()
    .sum()
)


seqn_duplicates = int(
    df["SEQN"]
    .duplicated()
    .sum()
)


seqn_min = df["SEQN"].min()
seqn_max = df["SEQN"].max()


print(
    "Missing:",
    seqn_missing
)

print(
    "Duplicates:",
    seqn_duplicates
)

print(
    "Minimum:",
    seqn_min
)

print(
    "Maximum:",
    seqn_max
)


if seqn_missing != 0:
    raise ValueError(
        "Unexpected missing SEQN."
    )


if seqn_duplicates != 0:
    raise ValueError(
        "Duplicate SEQN values found."
    )


if seqn_min != 31128:
    raise ValueError(
        f"Unexpected minimum SEQN: {seqn_min}"
    )


if seqn_max != 41474:
    raise ValueError(
        f"Unexpected maximum SEQN: {seqn_max}"
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 12. CDC CHECK - MCQ150Q
# ============================================================

#
# School days missed.
#
# CDC:
#
# 0-180   = 3,208
# 99999   = 2
# Missing = 6,612
#
# XPT contains 1,028 zero responses.
#

s = df["MCQ150Q"]


if not (
    int(
        s.between(
            0,
            180
        ).sum()
    ) == 3208

    and

    int(
        s.eq(0)
        .sum()
    ) == 1028

    and

    int(
        s.eq(99999)
        .sum()
    ) == 2

    and

    int(
        s.isna()
        .sum()
    ) == 6612
):

    raise ValueError(
        "MCQ150Q differs from CDC/source."
    )


print(
    "\nPASS: MCQ150Q matches CDC/source."
)

print(
    "      Zero days = 1,028"
)


# ============================================================
# 13. CDC CHECK - MCQ180B
# ============================================================

#
# Age first told had congestive heart failure.
#
# CDC valid range begins at 0.
#

s = df["MCQ180B"]


if not (
    int(
        s.between(
            0,
            84
        ).sum()
    ) == 169

    and

    int(
        s.eq(0)
        .sum()
    ) == 1

    and

    int(
        s.eq(85)
        .sum()
    ) == 9

    and

    int(
        s.eq(99999)
        .sum()
    ) == 2

    and

    int(
        s.isna()
        .sum()
    ) == 9642
):

    raise ValueError(
        "MCQ180B differs from CDC/source."
    )


print(
    "PASS: MCQ180B matches CDC/source."
)

print(
    "      Valid zero age = 1"
)


# ============================================================
# 14. CDC CHECK - MCQ180L
# ============================================================

#
# Age first told had liver condition.
#
# CDC valid range = 0-82.
#

s = df["MCQ180L"]


if not (
    int(
        s.between(
            0,
            82
        ).sum()
    ) == 163

    and

    int(
        s.eq(0)
        .sum()
    ) == 1

    and

    int(
        s.isna()
        .sum()
    ) == 9659
):

    raise ValueError(
        "MCQ180L differs from CDC/source."
    )


print(
    "PASS: MCQ180L matches CDC/source."
)

print(
    "      Valid zero age = 1"
)


# ============================================================
# 15. CDC CHECK - MCQ245B
# ============================================================

#
# Number of workdays missed.
#
# CDC:
#
# 0-365   = 3,842
# 99999   = 2
# Missing = 5,978
#
# XPT contains 2,061 zeros.
#

s = df["MCQ245B"]


if not (
    int(
        s.between(
            0,
            365
        ).sum()
    ) == 3842

    and

    int(
        s.eq(0)
        .sum()
    ) == 2061

    and

    int(
        s.eq(99999)
        .sum()
    ) == 2

    and

    int(
        s.isna()
        .sum()
    ) == 5978
):

    raise ValueError(
        "MCQ245B differs from CDC/source."
    )


print(
    "PASS: MCQ245B matches CDC/source."
)

print(
    "      Zero workdays = 2,061"
)


# ============================================================
# 16. CDC CHECK - MCQ340
# ============================================================

#
# Number of PSA tests in last 5 years.
#
# CDC:
#
# 0-20    = 799
# 999     = 32
# Missing = 8,991
#
# XPT contains 48 zeros.
#

s = df["MCQ340"]


if not (
    int(
        s.between(
            0,
            20
        ).sum()
    ) == 799

    and

    int(
        s.eq(0)
        .sum()
    ) == 48

    and

    int(
        s.eq(999)
        .sum()
    ) == 32

    and

    int(
        s.isna()
        .sum()
    ) == 8991
):

    raise ValueError(
        "MCQ340 differs from CDC/source."
    )


print(
    "PASS: MCQ340 matches CDC/source."
)

print(
    "      Zero PSA tests = 48"
)


# ============================================================
# 17. IMPORTANT PROJECT VARIABLE - MCQ220
# ============================================================

#
# Ever told you had cancer or malignancy.
#
# CDC:
#
# 1 Yes        = 414
# 2 No         = 4,561
# 9 Don't know = 4
# Missing      = 4,843
#

EXPECTED_MCQ220 = {
    1: 414,
    2: 4561,
    9: 4
}


actual_mcq220 = (
    df["MCQ220"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual_mcq220 = {
    int(k): int(v)
    for k, v
    in actual_mcq220.items()
}


if actual_mcq220 != EXPECTED_MCQ220:

    raise ValueError(
        "MCQ220 frequencies differ from CDC.\n"
        f"Actual: {actual_mcq220}"
    )


if int(
    df["MCQ220"]
    .isna()
    .sum()
) != 4843:

    raise ValueError(
        "MCQ220 missing count differs."
    )


print(
    "\nPASS: MCQ220 matches CDC."
)

print(
    "      Yes = 414"
)

print(
    "      No = 4,561"
)

print(
    "      Don't know = 4"
)

print(
    "      Missing = 4,843"
)


# ============================================================
# 18. EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# Expected source =
#
# original XPT
# +
# ONLY:
#
# MCQ150Q tiny -> 0
# MCQ180B tiny -> 0
# MCQ180L tiny -> 0
# MCQ245B tiny -> 0
# MCQ340  tiny -> 0
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 19. CONFIRM ALL FIELDS ARE WHOLE NUMBERS
# ============================================================

for col in df.columns:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if int(
        fractional.sum()
    ) != 0:

        raise ValueError(
            f"STOP: {col} contains genuine decimal values."
        )


print(
    "\nPASS: All 89 MCQ_D fields "
    "are exact whole-number fields."
)


# ============================================================
# 20. CLEAN SEQN + QUESTIONNAIRE FORMATTING
# ============================================================

#
# Examples:
#
# SEQN
# 31128.0 -> 31128
#
# MCQ010
# 1.0 -> 1
#
# MCQ025
# 99999.0 -> 99999
#
# MCQ150Q
# 0.0 -> 0
#
# MCQ245B
# 0.0 -> 0
#
# Missing
# NaN -> blank
#

for col in df.columns:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: SEQN and questionnaire fields "
    "converted to clean integer formatting."
)


# ============================================================
# 21. VALUE FIDELITY BEFORE EXPORT
# ============================================================

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


pre_export_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: Values match expected XPT "
    "after ONLY the five validated zero restorations."
)


# ============================================================
# 22. EXPORT CSV
# ============================================================

#
# NO:
#
# float_format
# round()
# recalculation
#
# Missing SAS numeric values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 23. RE-READ CSV WITH ROUND-TRIP PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 24. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 25. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- XPT -> CSV NUMERIC FIDELITY ---")


numeric_failures = []
total_numeric_differences = 0


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Total numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(numeric_failures)
    )


print(
    "PASS: All 89 CSV columns numerically "
    "match expected XPT values."
)


# ============================================================
# 26. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    diff = int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


    missing_mismatches += diff


    if diff > 0:

        missing_failures.append(
            (
                col,
                diff
            )
        )


print(
    "\nMissing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing-value positions changed:\n"
        + str(missing_failures)
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 27. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)
    raw_rows = list(reader)


if len(raw_rows) != 9822:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 28. STRICT RAW FORMAT FIDELITY
# ============================================================

#
# All MCQ_D fields are whole-number fields.
#
# Expected:
#
# ""              if missing
# str(int(value)) otherwise
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in expected_df.columns:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "Examples:"
    )


    for example in raw_examples:

        print(
            example
        )


    raise ValueError(
        "MCQ_D raw CSV formatting differs."
    )


print(
    "PASS: Raw CSV formatting matches "
    "whole-number questionnaire representation."
)


# ============================================================
# 29. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(
                dot_zero_examples
            ) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nValues ending '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        "Examples:",
        dot_zero_examples
    )

    raise ValueError(
        "Unnecessary .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 30. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = (
            value.lower()
        )


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        "Examples:",
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 31. FINAL TINY-ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact found in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 32. FINAL ZERO CHECK
# ============================================================

print("\n--- FINAL CSV ZERO CHECK ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        csv_df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col:<10}: "
        f"{actual_count:,}"
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: CSV zero count changed."
        )


print(
    "PASS: Final CSV zero restorations remain correct."
)


# ============================================================
# 33. FINAL MCQ220 CHECK
# ============================================================

final_mcq220 = (
    csv_df["MCQ220"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


final_mcq220 = {
    int(k): int(v)
    for k, v
    in final_mcq220.items()
}


if final_mcq220 != EXPECTED_MCQ220:

    raise ValueError(
        "MCQ220 frequencies changed after export."
    )


if int(
    csv_df["MCQ220"]
    .isna()
    .sum()
) != 4843:

    raise ValueError(
        "MCQ220 missing count changed."
    )


print(
    "PASS: Final MCQ220 frequencies match CDC."
)


# ============================================================
# 34. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("MCQ_D CONVERSION COMPLETE")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "\nValidated tiny -> zero:"
)

print(
    "MCQ150Q:",
    1028
)

print(
    "MCQ180B:",
    1
)

print(
    "MCQ180L:",
    1
)

print(
    "MCQ245B:",
    2061
)

print(
    "MCQ340:",
    48
)

print(
    "TOTAL:",
    EXPECTED_TINY_TOTAL
)

print(
    "\nGenuine decimal variables:",
    0
)

print(
    "SEQN/code/count formatting:",
    "integer"
)

print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Values ending .0:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)

print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "MCQ_D_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2005-2006 MCQ_D XPT -> CSV
Rows: 9,822
Columns: 89
PASS: MCQ_D structure = 9,822 rows x 89 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 3,139

Tiny values by variable:
MCQ245B    2061
MCQ150Q    1028
MCQ340       48
MCQ180L       1
MCQ180B       1

PASS: Exactly 3,139 validated tiny artifacts found.

--- ZERO RESTORATION ---
MCQ150Q   : 1,028 tiny -> 0
MCQ180B   : 1 tiny -> 0
MCQ180L   : 1 tiny -> 0
MCQ245B   : 2,061 tiny -> 0
MCQ340    : 48 tiny -> 0

--- RESTORED ZERO COUNTS ---
MCQ150Q   : 1,028
MCQ180B   : 1
MCQ180L   : 1
MCQ245B   : 2,061
MCQ340    : 48
PASS: All 3,139 validated zeros restored.

Remaining tiny artifacts: 0
PASS: No tiny-value artifacts remain.

--- FRACTIONAL VALUE CHECK ---
PASS: MCQ_D contains no genuine decimal variables.

--- SEQN CHECK ---
Missing: 0
Duplicates: 0
Minimum: 31128.0
Maximum: 41474.0
PASS: SEQN validated.

PASS: MCQ150Q matches CDC/source.
      Zero days = 1,028
PASS: MCQ180B matches CDC/sour